# **Exponential CURVE FIT function algorithm for chromosome_to_chromosome distance**
This script fits an exponential curve to the mean of each cell type of the *C. elegans* embryo in batch process. This curve uses the exponential curve fitting function where $y = a * (1 - e^{-x/b})$. The goal for fitting a mathematical function to the mean value is to determine the *Final chromosome-to-chromosome length* and *Segregation speed* at the required time point.

#### `INPUT FILES` 
The csv files of each cell type containing the chromosome-to-chromosome distance data of different experiments (n-value). 

#### `OUTPUT FILES` 
* **FIRST_GROUP_OUTPUT_FILES**: The *.png* files of the fitted plot of each cell type. 

* **SECOND_GROUP_OUTPUT_FILE**: A *.csv* file containing the **Final chromosomes length (µm)** and the **Segregation speed (µm/minute)** of each of the cell type. 

In [1]:
# library packages
import os
import warnings
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from numpy import exp, linspace, random, arange
from scipy.optimize import curve_fit, least_squares

In [2]:
# input folder
folder = r'D:\data\Analysis Data\python_analysis\input'

# output folder
save_files = r'D:\data\Analysis Data\python_analysis\output'

In [3]:
# new DataFrame to append the new generated table 
fit_Result = pd.DataFrame()

# read out individual files and compute for different operations for each file
for file in os.scandir(folder):
    df = pd.read_csv(file)
    
    # create a new dataframe 
    Exp_Column = df.loc[:, df.columns.str.startswith('Exp')]
    mean_column = df.loc[:, df.columns.str.startswith('mean')]
    time_column = df.loc[:, df.columns.str.startswith('time')]
    
    # define the start and end time points to compute
    x_start = np.where(time_column >= 0)[0][0]
    x_end = np.where((time_column <= 154.5) & (time_column == 154.5))[0][0] + 1
    
    # compute the time frames for column values
    new_x = time_column[x_start:x_end]
    new_exp = Exp_Column[x_start:x_end]
    new_mean = mean_column[x_start:x_end]
    
    # create a new dataframe
    df_Table = pd.concat([new_x, new_exp, new_mean], axis=1)
    
    '''
    drop all the rows were n-value is less than 3 the mean and the time 
    columns are included, ie, the number of rows to be computed should be >=3
    '''
    newTable = df_Table.dropna(thresh=3) 
    
    # Define the exponential function to fit in the data
    '''
    a = initial amplitude of the function
    b = time constant (the time taken for the function to reach approximately 63.2% of its maximum value, 
    i.e., 1-1/e as x approaches infinity)
    '''
    def exponential(x, a, b):
        return a * (1 - np.exp(-x / b))
    
    # ignore warning
    warnings.filterwarnings("ignore")
    
    # define the plot dimension 
    plt.figure(figsize=(5,5))
    
    for i_col in newTable.columns[1:-1]:
        # use try: - except: function in a situation where the .dropna(thresh=3) is  not met
        try:
            NaN_table = newTable[['time', i_col]]
            plot_table = NaN_table.dropna()

            # x and y variables
            x = plot_table['time']
            y = plot_table[i_col]
            
            # compute the initial guess
            initial_guess = [10, 50]

            # summarize the parameter
            popt, pcov = curve_fit(exponential, x, y, initial_guess, maxfev=10000)

            ''' 
            Define a sequence of inputs between the smallest and largest known inputs and define the fit. 
            Let the maximum input assume the maximum values of x. 
            '''
            x_fit = np.arange(0, x.max()+0.1, 0.1, dtype=None)
            y_fit = exponential(x_fit, *popt)
            
            # primary plot
            ax1 = sns.scatterplot(x=x, y=y, alpha=0.2)

            # fit on each plot 
            sns.lineplot(x_fit, y_fit, alpha = 1, ax=ax1) 
            
            # add the desired features on the plot
            ax1.set_xlabel('time [s]', fontsize= 18)
            ax1.set_ylabel('distance [μm]', fontsize= 18)
            plot_title = (file.name).split('.')[0]
            ax1.axes.set_title(plot_title, fontsize= 20, fontweight='bold')
            
            #save plots
            plotfile = (file.name).split('.')[0] + '.png'
            plt.savefig(os.path.join(save_files, plotfile), dpi=300)
            
             # assign variables to the fit parameter output
            amp = popt[0]
            T = popt[1]

            # determine the slope == segregation speed
            segregation_speed = (amp/T)*60 # convert to µm/min by multiplying by 60

            # determine the maximum segregation length == final chromosome_to_chromosome distance
            final_C_C_length = amp

            # create a table for the speed of segregation and the maximum distance
            parameters = {'Final chromosomes length (µm)': final_C_C_length, 
                          'Segregation speed (µm/min)': segregation_speed}
            parameters_df = pd.DataFrame.from_dict(parameters, orient='index', columns=[file.name.split('.')[0]])
            fit_Result = pd.concat([fit_Result, parameters_df], axis=1)   
            
        except Exception:
            pass
        
# transpose the table
fit_Result_transpose = (fit_Result).T

# assign header to the index column
fit_Result_transpose.index.names = ['Cells']

# save the table, fit_Result, to a csv file
fit_Result_transpose.to_csv(os.path.join(save_files, 'Fit_Result_chromosome_chromosome.csv'))

# close all open windows
plt.close('all')
